# Lab: Encoder-Decoder for Sequence-to-Sequence Learning

**Aim:** Build an encoder-decoder RNN model for sequence-to-sequence tasks (English to French translation on a small dataset).

In [1]:
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np

torch.manual_seed(42)
random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


## Small English-French Dataset

In [2]:
pairs = [
    ('i am cold', 'j ai froid'),
    ('you are tired', 'tu es fatigue'),
    ('he is hungry', 'il a faim'),
    ('she is happy', 'elle est heureuse'),
    ('we are ready', 'nous sommes prets'),
    ('they are sad', 'ils sont tristes'),
    ('i love you', 'je t aime'),
    ('he runs fast', 'il court vite'),
    ('she sings well', 'elle chante bien'),
    ('the cat is cute', 'le chat est mignon'),
    ('the dog runs fast', 'le chien court vite'),
    ('i am happy', 'je suis heureux'),
    ('he is cold', 'il a froid'),
    ('you are happy', 'tu es heureux'),
    ('we love france', 'nous aimons la france'),
    ('i am tired', 'je suis fatigue'),
    ('she is cold', 'elle a froid'),
    ('he is ready', 'il est pret'),
    ('they run fast', 'ils courent vite'),
    ('i sing well', 'je chante bien'),
]

SOS, EOS, PAD = '<sos>', '<eos>', '<pad>'

def build_vocab(sents):
    words = set()
    for s in sents:
        for w in s.split():
            words.add(w)
    w2i = {PAD: 0, SOS: 1, EOS: 2}
    for w in sorted(words):
        if w not in w2i:
            w2i[w] = len(w2i)
    return w2i, {v: k for k, v in w2i.items()}

src_vocab, src_idx2w = build_vocab([p[0] for p in pairs])
tgt_vocab, tgt_idx2w = build_vocab([p[1] for p in pairs])

print(f'Source vocabulary size: {len(src_vocab)}')
print(f'Target vocabulary size: {len(tgt_vocab)}')
print(f'Training pairs        : {len(pairs)}')

Source vocabulary size: 30
Target vocabulary size: 39
Training pairs        : 20


## Encoder and Decoder Modules

In [3]:
def tensorize(sentence, vocab):
    ids = [vocab.get(w, 0) for w in sentence.split()]
    ids.append(vocab[EOS])
    return torch.tensor(ids, dtype=torch.long).to(device)


class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_size, batch_first=True)

    def forward(self, x):
        emb = self.embedding(x.unsqueeze(0))
        _, hidden = self.gru(emb)
        return hidden


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRUCell(embed_dim, hidden_size)
        self.fc  = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        emb = self.embedding(x.unsqueeze(0)).squeeze(0)  # (1, embed)
        h = self.gru(emb, hidden.squeeze(0))
        logits = self.fc(h)
        return logits, h.unsqueeze(0)


EMBED, HIDDEN = 32, 64
encoder = Encoder(len(src_vocab), EMBED, HIDDEN).to(device)
decoder = Decoder(len(tgt_vocab), EMBED, HIDDEN).to(device)

enc_params = sum(p.numel() for p in encoder.parameters())
dec_params = sum(p.numel() for p in decoder.parameters())
print(f'Encoder params: {enc_params:,}')
print(f'Decoder params: {dec_params:,}')
print(f'Total          : {enc_params + dec_params:,}')

Encoder params: 19,776
Decoder params: 22,599
Total          : 42,375


## Training

In [4]:
enc_opt = optim.Adam(encoder.parameters(), lr=0.01)
dec_opt = optim.Adam(decoder.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

TF_RATIO = 0.5
loss_hist = []

for epoch in range(1, 201):
    total_loss = 0
    random.shuffle(pairs)
    for src_s, tgt_s in pairs:
        src_t = tensorize(src_s, src_vocab)
        tgt_t = tensorize(tgt_s, tgt_vocab)

        enc_opt.zero_grad(); dec_opt.zero_grad()

        hidden = encoder(src_t)
        inp = torch.tensor([tgt_vocab[SOS]], device=device)

        loss = torch.tensor(0.0, device=device)
        for t in range(len(tgt_t)):
            logits, hidden = decoder(inp, hidden)
            loss += loss_fn(logits, tgt_t[t].unsqueeze(0))
            # teacher forcing
            inp = tgt_t[t] if random.random() < TF_RATIO else logits.argmax(1)

        loss.backward()
        enc_opt.step(); dec_opt.step()
        total_loss += loss.item() / len(tgt_t)

    avg = total_loss / len(pairs)
    loss_hist.append(avg)
    if epoch % 40 == 0:
        print(f'Epoch {epoch:3d} | Loss: {avg:.4f}')

print('Training done.')

Epoch  40 | Loss: 0.0028
Epoch  80 | Loss: 0.0008
Epoch 120 | Loss: 0.0004
Epoch 160 | Loss: 0.0002
Epoch 200 | Loss: 0.0001
Training done.


## Translation Function

In [5]:
def translate(sentence, max_len=12):
    encoder.eval(); decoder.eval()
    with torch.no_grad():
        src_t = tensorize(sentence, src_vocab)
        hidden = encoder(src_t)
        inp = torch.tensor([tgt_vocab[SOS]], device=device)
        words = []
        for _ in range(max_len):
            logits, hidden = decoder(inp, hidden)
            pred = logits.argmax(1).item()
            if pred == tgt_vocab[EOS]:
                break
            words.append(tgt_idx2w.get(pred, '<unk>'))
            inp = torch.tensor([pred], device=device)
    return ' '.join(words)

print('Translation Results:')
print('-' * 50)
test_sentences = [
    'i am cold', 'she is happy', 'he runs fast',
    'you are tired', 'i love you', 'the cat is cute'
]
for s in test_sentences:
    pred = translate(s)
    pairs_dict = dict(pairs)
    ref = pairs_dict.get(s, '?')
    match = '✓' if pred == ref else '~'
    print(f'  {s:<22} -> {pred:<25} (ref: {ref}) {match}')

Translation Results:
--------------------------------------------------
  i am cold              -> j ai froid              (ref: j ai froid) V
  she is happy           -> elle est heureuse       (ref: elle est heureuse) V
  he runs fast           -> il court vite           (ref: il court vite) V
  you are tired          -> tu es fatigue           (ref: tu es fatigue) V
  i love you             -> je t aime               (ref: je t aime) V
  the cat is cute        -> le chat est mignon      (ref: le chat est mignon) V


## BLEU Score

In [6]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
nltk.download('punkt', quiet=True)

smooth = SmoothingFunction().method1
scores = []
for src_s, tgt_s in pairs:
    pred = translate(src_s).split()
    ref  = [tgt_s.split()]
    scores.append(sentence_bleu(ref, pred, smoothing_function=smooth))

print(f'Average BLEU on training set: {np.mean(scores):.4f}')
print(f'Min BLEU: {min(scores):.4f}  |  Max BLEU: {max(scores):.4f}')

Average BLEU on training set: 0.6280
Min BLEU: 0.5623  |  Max BLEU: 1.0000


## Observations

The encoder compresses the full input sequence into a single context vector (the final hidden state), and the decoder generates the target sequence conditioned on that vector. This architecture works well for short sentences but struggles with long ones because all information must fit in a fixed-size vector. That is exactly the motivation for attention mechanisms — letting the decoder look at all encoder hidden states at each step rather than just the final one.